# 05 - Where does the LLM signal come from? Breakdowns, ablations and a leakage check

Notebook 04 asks whether the LLM signal makes money. This notebook asks **why it does or does not**, and whether we can
trust a positive result if we see one.

| Step | What it answers |
|---|---|
| 1. Load | The LLM scores we have, next to FinBERT on exactly the same rows |
| 2. Event type | Does the signal work for some kinds of news (earnings, M&A, ...) and not others? |
| 3. Ablations | Which part of the LLM output carries the information: sign, magnitude, confidence? |
| 4. Sub-periods | Is it the same before, during and after the March 2020 crash? |
| 5. Leakage check | Does the model use its *memory* of the company (and of 2020) rather than the headline text? |
| 6. Errors | The most confident wrong calls, read by eye |

**Read this first.** The test period is 2020-01-14 to 2020-06-10. The model was trained long after that, so it may have
seen articles about what happened to these stocks. That is **look-ahead through the model's memory**, and the price
alignment in notebook 01 cannot fix it. Section 5 can only *reduce and measure* it (by hiding company names); it cannot
remove it (the model may still recognise an event such as "stimulus bill fails senate vote"). A clean test would need news
published after the model's training cutoff, which we do not have here.

**Everything below is provisional until notebook 03 has scored the full target sample.** The row counts printed in
section 1 say how much of it we have.

In [1]:
import sys; sys.path.append('..')
import re, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from src.evaluate import cross_section_ic, subset_cross_section_ic, pooled_ic

SEED = 42
np.random.seed(SEED)
PROCESSED = Path('../data/processed')
RAW = Path('../data/raw')
pd.set_option('display.width', 220)

N_BOOT = 1000             # day-cluster bootstrap resamples
RUN_LLM_MASKED = True     # sends masked headlines to the LLM; answers are cached, so a re-run costs nothing once they exist
N_MASKED = 300            # how many masked headlines to re-score

## 1. Load the LLM scores next to FinBERT

In [2]:
ds = pd.read_csv(PROCESSED / 'dataset.csv', parse_dates=['entry_date'])
ds['row_id'] = ds.index
bs = pd.read_csv(PROCESSED / 'baseline_scores.csv')
assert len(bs) == len(ds) and (bs['row_id'].values == ds['row_id'].values).all(), 'score file does not line up with the dataset'
ds = ds.merge(bs[['row_id', 'split', 'finbert']], on='row_id', how='left')
test = ds[ds['split'] == 'test']

candidates = [PROCESSED / 'llm_scores_test_zeroshot_batch.csv', PROCESSED / 'llm_scores_test_zeroshot.csv']
llm_file = next((f for f in candidates if f.exists()), None)
assert llm_file is not None, 'run notebook 03 first: no LLM score file found'
lb = pd.read_csv(llm_file)
ok = lb[lb['ok'].astype(bool)]
llm = test.merge(ok[['row_id', 'sentiment', 'score', 'event_type', 'confidence']].rename(columns={'score': 'llm_score'}), on='row_id')
assert llm['row_id'].is_unique

print(f'score file: {llm_file.name}')
print(f'test rows: {len(test):,} | with a valid LLM answer: {len(llm):,} ({len(llm) / len(test):.0%}) | '
      f'days: {llm["entry_date"].nunique()} of {test["entry_date"].nunique()} | tickers: {llm["ticker"].nunique()}')
print('\nLLM labels in the sample:')
print(llm['sentiment'].value_counts(normalize=True).round(3).to_string())
print('\nconfidence values:')
print(llm['confidence'].round(2).value_counts().sort_index().to_string())

score file: llm_scores_test_zeroshot_batch.csv
test rows: 7,542 | with a valid LLM answer: 3,000 (40%) | days: 106 of 107 | tickers: 100

LLM labels in the sample:
sentiment
positive    0.359
negative    0.330
neutral     0.311

confidence values:
confidence
0.10       1
0.20       3
0.30      31
0.40      34
0.50     100
0.60     222
0.70     539
0.80    1164
0.85      24
0.90     877
0.95       5


In [3]:
# reference line: the overall signal quality on this sample, with FinBERT scored on exactly the same rows
rows = []
for name, col in [('LLM score', 'llm_score'), ('FinBERT', 'finbert')]:
    r = cross_section_ic(llm, col, n_boot=N_BOOT, seed=SEED)
    rows.append({'signal': name, 'rows': r['n'], 'days': r['n_days'], 'pooled IC': pooled_ic(llm, col),
                 'within-day IC': r['ic'], 'CI low': r['lo'], 'CI high': r['hi']})
display(pd.DataFrame(rows).set_index('signal').round(4))

,rows,days,pooled IC,within-day IC,CI low,CI high
signal,,,,,,
LLM score,3000,106,0.0035,0.0297,-0.0220,0.0823
FinBERT,3000,106,0.0048,0.0298,-0.0178,0.0764


**What "within-day IC" means here.** Notebook 02 used a pooled IC and a mean of per-day ICs. For the breakdowns below
there are too few stocks per day *inside one subgroup* for a per-day IC, so we use a third version. Subtract each day's mean
from both the score and the abnormal return, then rank-correlate across all rows:

$$
\tilde s_{i,d} = s_{i,d} - \bar s_d, \qquad \tilde y_{i,d} = y_{i,d} - \bar y_d, \qquad
\mathrm{IC}_{\text{within}} = \rho_{\text{Spearman}}\big(\tilde s, \tilde y\big).
$$

Why bother: $y$ is the return *minus SPY*, but a high-beta stock still loses more than SPY on a crash day, and the LLM
scores are also lower that day. A plain pooled IC would then partly measure "bad market day". Removing the day mean keeps only the
comparison *between stocks on the same day*. The confidence interval resamples **whole days** (a cluster bootstrap), because
headlines from the same day are not independent. The check right below runs it on fake data where we know the answer.

In [4]:
# fake data with a known answer: 80 days x 30 stocks
rng = np.random.default_rng(0)
days = pd.date_range('2020-01-14', periods=80, freq='B')
fake = pd.DataFrame({'entry_date': np.repeat(days, 30)})
day_effect = np.repeat(rng.normal(size=80), 30)
fake['abn_ret'] = day_effect * 0.01 + rng.normal(size=len(fake)) * 0.01
fake['noise'] = rng.normal(size=len(fake))                                     # no information
fake['day_only'] = day_effect + rng.normal(size=len(fake)) * 0.3               # follows the market day, nothing stock-specific
fake['real'] = (fake['abn_ret'] - fake.groupby('entry_date')['abn_ret'].transform('mean')) * 100 + rng.normal(size=len(fake)) * 2
rows = []
for c in ['noise', 'day_only', 'real']:
    r = cross_section_ic(fake, c, n_boot=300, seed=SEED)
    rows.append({'fake signal': c, 'pooled IC': pooled_ic(fake, c), 'within-day IC': r['ic'], 'CI low': r['lo'], 'CI high': r['hi']})
chk = pd.DataFrame(rows).set_index('fake signal')
display(chk.round(3))
assert chk.loc['noise', 'CI low'] < 0 < chk.loc['noise', 'CI high']          # noise: interval contains 0
assert chk.loc['day_only', 'pooled IC'] > 0.4 and abs(chk.loc['day_only', 'within-day IC']) < 0.1   # day effect removed
assert chk.loc['real', 'CI low'] > 0.2                                         # a real within-day signal is found
print('within-day IC checks: OK')

,pooled IC,within-day IC,CI low,CI high
fake signal,,,,
noise,0.014,0.007,-0.033,0.052
day_only,0.648,0.002,-0.036,0.040
real,0.282,0.426,0.398,0.460


within-day IC checks: OK


## 2. By event type

The LLM also labels each headline with an event type. If the signal exists at all, we would expect it in the types where the
text carries clear information (earnings beats and misses, guidance) and not in vague ones (`other`, `macro`).

**Caution on reading this table.** With 8 event types, even a signal that is pure noise will show about one "significant"
interval in 20 tries by chance. Small groups also have wide intervals. We look for a *pattern that makes sense*, not for the one
interval that excludes 0.

In [5]:
rows = []
for et, g in llm.groupby('event_type'):
    mask = llm['event_type'] == et
    r_llm = subset_cross_section_ic(llm, mask, 'llm_score', n_boot=N_BOOT, seed=SEED)
    r_fb = subset_cross_section_ic(llm, mask, 'finbert', n_boot=N_BOOT, seed=SEED)
    rows.append({'event type': et, 'rows': len(g), 'share': len(g) / len(llm), 'days': g['entry_date'].nunique(),
                 'LLM IC': r_llm['ic'], 'LLM low': r_llm['lo'], 'LLM high': r_llm['hi'],
                 'FinBERT IC': r_fb['ic'], 'FinBERT low': r_fb['lo'], 'FinBERT high': r_fb['hi']})
by_event = pd.DataFrame(rows).set_index('event type').sort_values('rows', ascending=False)
display(by_event.round(3))

# rough power: the smallest correlation a group of n rows can tell from 0 is about 2 / sqrt(n)
by_event['detectable |IC| ~'] = 2 / np.sqrt(by_event['rows'])
print('rule of thumb for the smallest IC each group could detect (about 2/sqrt(rows)):')
print(by_event['detectable |IC| ~'].round(3).to_string())

,rows,share,days,LLM IC,LLM low,LLM high,FinBERT IC,FinBERT low,FinBERT high
event type,,,,,,,,,
other,1140,0.380,106,0.045,-0.033,0.121,0.057,-0.004,0.117
macro,587,0.196,101,0.011,-0.077,0.103,0.032,-0.049,0.116
analyst_rating,569,0.190,103,-0.052,-0.157,0.051,-0.037,-0.132,0.061
product,256,0.085,90,0.088,-0.054,0.232,0.053,-0.091,0.186
earnings,254,0.085,78,0.110,-0.035,0.243,0.027,-0.098,0.139
guidance,75,0.025,52,0.122,-0.133,0.359,0.052,-0.175,0.279
legal,71,0.024,47,-0.078,-0.354,0.179,-0.199,-0.472,0.100
m&a,48,0.016,31,0.123,-0.168,0.395,0.044,-0.250,0.336


rule of thumb for the smallest IC each group could detect (about 2/sqrt(rows)):
event type
other             0.059
macro             0.083
analyst_rating    0.084
product           0.125
earnings          0.125
guidance          0.231
legal             0.237
m&a               0.289


## 3. Ablations: which part of the LLM answer carries the information?

The LLM returns a label, a score in $[-1,1]$, an event type and a confidence in $[0,1]$. We try several ways to turn that
into one number, and also look at the rows the model was sure about:

| Variant | Definition |
|---|---|
| score | the LLM score as is |
| sign only | $\operatorname{sign}(s)$, i.e. only the direction, no magnitude |
| score x confidence | $s \cdot c$, to down-weight guesses |
| confident rows only | the score, restricted to rows with $c \ge 0.8$ |
| non-neutral rows only | the score, restricted to rows where $s \ne 0$ |

If "sign only" does as well as "score", the magnitude adds nothing and the model is effectively a 3-way classifier. If
"confident rows only" is clearly better, confidence is useful. These variants are all computed from the same answers, so they
are **not independent tests**: do not count each as a separate confirmation.

In [6]:
llm = llm.copy()
llm['sign_only'] = np.sign(llm['llm_score'])
llm['score_x_conf'] = llm['llm_score'] * llm['confidence']

variants = [
    ('score',                 'llm_score',    pd.Series(True, index=llm.index)),
    ('sign only',             'sign_only',    pd.Series(True, index=llm.index)),
    ('score x confidence',    'score_x_conf', pd.Series(True, index=llm.index)),
    ('confident rows only',   'llm_score',    llm['confidence'] >= 0.8),
    ('non-neutral rows only', 'llm_score',    llm['llm_score'] != 0),
    ('FinBERT (same rows)',   'finbert',      pd.Series(True, index=llm.index)),
]
rows = []
for name, col, mask in variants:
    r = subset_cross_section_ic(llm, mask, col, n_boot=N_BOOT, seed=SEED)
    rows.append({'variant': name, 'rows': r['n'], 'days': r['n_days'], 'IC': r['ic'], 'CI low': r['lo'], 'CI high': r['hi']})
display(pd.DataFrame(rows).set_index('variant').round(4))

,rows,days,IC,CI low,CI high
variant,,,,,
score,3000,106,0.0297,-0.0220,0.0823
sign only,3000,106,0.0179,-0.0306,0.0701
score x confidence,3000,106,0.0294,-0.0229,0.0821
confident rows only,2070,106,0.0314,-0.0290,0.0933
non-neutral rows only,2145,106,0.0360,-0.0192,0.0954
FinBERT (same rows),3000,106,0.0298,-0.0178,0.0764


## 4. Sub-periods: before, during and after the crash

Three periods, **chosen before looking at results**: January 14 to February 29 (before the crash), March (the crash and the
first rebound) and April 1 to June 10. If the signal only works in March, either it is a crash-regime effect (fear headlines
followed by large moves) or the model remembers March 2020. This table cannot tell the two apart, but a signal that exists
in only one period is not something to build on.

In [7]:
periods = [('Jan 14 - Feb 29', '2020-01-14', '2020-02-29'),
           ('March',           '2020-03-01', '2020-03-31'),
           ('Apr 1 - Jun 10',  '2020-04-01', '2020-06-10')]
rows = []
for label, a, b in periods:
    mask = (llm['entry_date'] >= a) & (llm['entry_date'] <= b)
    for name, col in [('LLM score', 'llm_score'), ('FinBERT', 'finbert')]:
        r = subset_cross_section_ic(llm, mask, col, n_boot=N_BOOT, seed=SEED)
        rows.append({'period': label, 'signal': name, 'rows': r['n'], 'days': r['n_days'], 'IC': r['ic'], 'CI low': r['lo'], 'CI high': r['hi']})
display(pd.DataFrame(rows).set_index(['period', 'signal']).round(4))

rows  days      IC  CI low  CI high
period          signal                                        
Jan 14 - Feb 29 LLM score   831    34  0.0081 -0.0974   0.1124
                FinBERT     831    34  0.0416 -0.0587   0.1294
March           LLM score   737    22  0.0894 -0.0123   0.1830
                FinBERT     737    22  0.0913 -0.0018   0.1656
Apr 1 - Jun 10  LLM score  1432    50  0.0010 -0.0741   0.0789
                FinBERT    1432    50 -0.0173 -0.0815   0.0460

## 5. Leakage check: hide the company

**The worry.** An LLM that has read about Apple, Boeing and Carnival in 2020 may "know" how their stocks did. If so, its score
tells us about the model's memory, not about whether the headline contained news. The headline text can carry the name in
two ways: the company name (*"American Airlines Group Q4 EPS misses..."*) and the ticker we pass to the model next to it.

**The test.** For a sample of headlines, replace the company name and ticker by *"the company"*, send **no ticker**, and
score again. Then compare the two answers on the same headlines:

- **Agreement:** how often do the original and masked answers have the same sign? If the model only reads the text, masking
  should change little. If it relies on the name, answers drift.
- **Information:** does the masked score still line up with the next-day abnormal return as well as the original?

**What this cannot do.** It hides names, not events. "Fed announces stimulus measures" is still the March 2020 Fed. And name
masking is imperfect (short names, nicknames, products such as "iPhone"); the notebook measures how often it changed
the headline and shows examples. The sample is also small (a few hundred rows), so the IC comparison can only detect a
large change. We say so in the results instead of pretending otherwise.

### 5a. Company names

Names come from Yahoo Finance (public data, via `yfinance`) and are cached in `data/raw/company_names.csv`. A name like
*"American Airlines Group, Inc."* is shortened to the part a headline would actually use (*"American Airlines"*).

In [8]:
NAMES_FILE = RAW / 'company_names.csv'

def fetch_names(tickers):
    import yfinance as yf
    out = []
    for t in tickers:
        name = None
        try:
            info = yf.Ticker(t).info
            name = info.get('longName') or info.get('shortName')
        except Exception as e:
            print(f'  {t}: no name ({type(e).__name__})')
        out.append({'ticker': t, 'name': name})
        time.sleep(0.2)
    return pd.DataFrame(out)

if NAMES_FILE.exists():
    names_df = pd.read_csv(NAMES_FILE)
else:
    names_df = fetch_names(sorted(test['ticker'].unique()))
    names_df.to_csv(NAMES_FILE, index=False)
names = names_df.dropna(subset=['name']).set_index('ticker')['name'].to_dict()
print(f'names found for {len(names)} of {test["ticker"].nunique()} test tickers')
print(names_df[names_df['name'].isna()]['ticker'].tolist() or 'none missing')

names found for 100 of 100 test tickers
none missing


In [9]:
# trailing corporate words that headlines usually drop: "Archer-Daniels-Midland Company" -> "Archer-Daniels-Midland"
SUFFIX = re.compile(r'(?:,?\s+(?:Inc|Incorporated|Corp|Corporation|Company|Co|Ltd|Limited|plc|PLC|LLC|Holdings?|Group|N\.V|S\.A|L\.P|Class [A-C]|Common Stock))+\.?$', re.I)
# a first word that many companies share is too risky to mask on its own ("American", "United", "First" ...)
GENERIC_FIRST = {'american', 'united', 'general', 'first', 'national', 'international', 'global', 'western', 'southern',
                 'northern', 'pacific', 'atlantic', 'capital', 'public', 'china', 'chinese', 'bank', 'new', 'north', 'south',
                 'east', 'west', 'great', 'real', 'home', 'digital', 'energy', 'health', 'financial', 'southwest', 'state'}
IS_FUND = re.compile(r'\b(?:ETF|Trust|Fund)\b', re.I)       # ETFs: their long names never appear in headlines, only the ticker does
# short names headlines use that cannot be derived from the legal name
EXTRA_ALIASES = {'LLY': ['Lilly'], 'GOOG': ['Google'], 'GOOGL': ['Google'], 'PG': ['P&G'], 'JNJ': ['J&J']}

def core_name(name):
    n = re.sub(r'^The\s+', '', name.strip(), flags=re.I)         # "The Procter & Gamble Company" -> "Procter & Gamble Company"
    while True:                                                  # strip repeatedly: "..., Inc." then "Group"
        new = SUFFIX.sub('', n).strip(' ,.')
        new = re.sub(r'\s+(?:&|and)$', '', new)                  # "Wells Fargo &" / "Eli Lilly and" lose the dangling joiner
        if new == n:
            return n
        n = new

def aliases(name, ticker=None):
    core = core_name(name)
    out = {core}
    if core.lower().endswith('.com'):
        out.add(core[:-4])                                       # "Amazon.com" -> "Amazon"
    first = core.split()[0] if core else ''
    if not IS_FUND.search(name) and len(first) >= 4 and first.isalpha() and first.lower() not in GENERIC_FIRST:
        out.add(first)                                           # "Archer" ... only for distinctive first words
    out.update(EXTRA_ALIASES.get(ticker, []))
    return sorted((a for a in out if len(a) >= 3), key=len, reverse=True)

def mask_company(headline, ticker, name_lookup):
    # replace the company name (and "'s") and the ticker by "the company"; returns the new text
    text = headline
    for alias in aliases(name_lookup.get(ticker, ''), ticker) if name_lookup.get(ticker) else EXTRA_ALIASES.get(ticker, []):
        # the alias may be followed by corporate words that the headline keeps ("... Airlines Group", "... Corp.")
        pattern = r"(?<![A-Za-z])" + re.escape(alias) + r"(?:\s+(?:Group|Inc|Corp|Corporation|Company|Co|Holdings?|Ltd|plc)\b\.?)*(?![A-Za-z])('s)?"
        text = re.sub(pattern, lambda m: 'the company' + (m.group(1) or ''), text, flags=re.I)
    if len(ticker) >= 2:                                         # one-letter tickers ("A", "T", "F") would match ordinary words
        text = re.sub(r"(?<![A-Za-z])\$?" + re.escape(ticker) + r"(?![A-Za-z])", 'the company', text)
    else:
        text = re.sub(r"\$" + re.escape(ticker) + r"(?![A-Za-z])", 'the company', text)
    return text

# unit checks on hand-made names (no network)
demo = {'AAL': 'American Airlines Group, Inc.', 'AMZN': 'Amazon.com, Inc.', 'A': 'Agilent Technologies, Inc.',
        'ADM': 'Archer-Daniels-Midland Company', 'LLY': 'Eli Lilly and Company', 'PG': 'The Procter & Gamble Company',
        'WFC': 'Wells Fargo & Company', 'GOOGL': 'Alphabet Inc.', 'XLE': 'State Street Energy Select Sector SPDR ETF'}
assert mask_company('American Airlines Group Q4 Adj. EPS $1.15 Misses', 'AAL', demo).startswith('the company Q4')
assert mask_company('Archer-Daniels-Midland Co. cuts view', 'ADM', demo) == 'the company cuts view'
assert 'Amazon' not in mask_company("Amazon's cloud unit wins deal; AMZN up", 'AMZN', demo)
assert mask_company('A Look At The Week Ahead', 'A', demo) == 'A Look At The Week Ahead'      # one-letter ticker: left alone
assert 'ADM' not in mask_company('ADM (ADM) beats; Archer-Daniels-Midland raises outlook', 'ADM', demo)
assert mask_company('Some headline about nothing', 'AAL', demo) == 'Some headline about nothing'
assert mask_company('Eli Lilly Q1 EPS Beats', 'LLY', demo) == 'the company Q1 EPS Beats'          # was not masked at all before
assert mask_company('Lilly Announces Trial Results', 'LLY', demo) == 'the company Announces Trial Results'
assert mask_company('Procter & Gamble Raises Outlook', 'PG', demo) == 'the company Raises Outlook'
assert mask_company('Wells Fargo Cuts Dividend', 'WFC', demo) == 'the company Cuts Dividend'      # was 'the company Fargo ...'
assert mask_company('Google Unveils New Pixel', 'GOOGL', demo) == 'the company Unveils New Pixel'
assert mask_company('State Of The Economy: XLE Falls', 'XLE', demo) == 'State Of The Economy: the company Falls'   # ETF: only the ticker
print('masking unit checks: OK')

masking unit checks: OK


### 5b. How much did masking change, and what does it look like?

We only re-score headlines that masking actually changed (if the text is identical, the answer is identical by construction,
and nothing is learned).

In [10]:
llm['masked'] = [mask_company(h, t, names) for h, t in zip(llm['headline'], llm['ticker'])]
llm['changed'] = llm['masked'] != llm['headline']
print(f'headlines changed by masking: {llm["changed"].sum():,} of {len(llm):,} ({llm["changed"].mean():.0%})')

# Independent residual check (it does NOT reuse the aliases, otherwise it could never find a name the aliases miss):
# a distinctive word of the company's own name that is still visible in the masked headline. ETFs are skipped because
# their names are generic words ("Japan", "Energy") that headlines legitimately use.
def name_words(name):
    return {w for w in re.findall(r'[A-Za-z]{4,}', core_name(name)) if w.lower() not in GENERIC_FIRST}

left = []
for m, t, ch in zip(llm['masked'], llm['ticker'], llm['changed']):
    if not names.get(t) or IS_FUND.search(names[t]):
        continue
    visible = [w for w in name_words(names[t]) if re.search(r'(?<![A-Za-z])' + re.escape(w) + r'(?![A-Za-z])', m, flags=re.I)]
    if visible:
        left.append((t, visible, m))
print(f'masked headlines that still show a distinctive word of the company name: {len(left)} of {len(llm):,} ({len(left) / len(llm):.1%})')
for t, visible, m in left[:5]:
    print(f'   [{t}] {visible}: {m[:110]}')

ex = llm[llm['changed']].sample(6, random_state=SEED)[['ticker', 'headline', 'masked']]
for _, r in ex.iterrows():
    print(f'\n[{r["ticker"]}]  {r["headline"]}\n   -> {r["masked"]}')

headlines changed by masking: 1,494 of 3,000 (50%)
masked headlines that still show a distinctive word of the company name: 74 of 3,000 (2.5%)
   [VZ] ['Communications']: What Comes First: Consumer Confidence Or Spending? Communications Sector Might Hold Clues
   [LUV] ['Airlines']: 3 ETFs To Consider For The Week Ahead: Big Earnings For Airlines, Chips And Consumers
   [UAL] ['Airlines']: 3 ETFs To Consider For The Week Ahead: Big Earnings For Airlines, Chips And Consumers
   [UAL] ['Airlines']: US Association Of Flight Attendants Alert Suggests Communications Works Of America Labor Union Has Called For 
   [VZ] ['Communications']: Huawei to Sue the company for Patent Infringement, Including Network Communications Technology

[REGN]  Regeneron Pharmaceuticals shares are trading higher after the company announced the FDA has accepted for priority review a biologics license application for REGN-EB3 to treat Ebola.
   -> the company shares are trading higher after the company announced t

### 5c. Re-score the masked headlines

A random sample of the changed headlines goes to the same model with the same batch prompt, **with no ticker**. Valid answers
are cached per headline, so an interrupted or rate-limited run resumes. `RUN_LLM_MASKED` is `False` by default because this uses
free-tier quota that the main scoring run needs; set it to `True` once notebook 03 has finished.

In [11]:
MASKED_FILE = PROCESSED / 'llm_masked_test.csv'
changed = llm[llm['changed']]
sample = changed.sample(min(N_MASKED, len(changed)), random_state=SEED) if len(changed) else changed

if RUN_LLM_MASKED and len(sample):
    from src import llm_extract
    try:
        out = llm_extract.extract_batch([(m, None) for m in sample['masked']], batch_size=10)
    except llm_extract.RateLimitExhausted as e:
        print('stopped by the daily limit; what was done is cached, run this cell again later.\n', e)
        out = None
    if out is not None:
        res = pd.DataFrame([{k: o.get(k) for k in ['ok', 'sentiment', 'score', 'event_type', 'confidence', 'error']} for o in out])
        res.insert(0, 'row_id', sample['row_id'].values)
        res.to_csv(MASKED_FILE, index=False)
        print(f'saved {MASKED_FILE.name}: {int(res["ok"].sum())} valid of {len(res)}')

if MASKED_FILE.exists():
    mk = pd.read_csv(MASKED_FILE)
    mk = mk[mk['ok'].astype(bool)].rename(columns={'score': 'masked_score', 'sentiment': 'masked_sentiment'})
    both = llm.merge(mk[['row_id', 'masked_score', 'masked_sentiment']], on='row_id')
    print(f'masked answers available: {len(both)}')
else:
    both = pd.DataFrame()
    print('no masked answers yet (set RUN_LLM_MASKED = True after notebook 03 has finished)')

saved llm_masked_test.csv: 300 valid of 300
masked answers available: 300


In [12]:
if len(both) >= 30:
    same_label = (both['sentiment'] == both['masked_sentiment']).mean()
    same_sign = (np.sign(both['llm_score']) == np.sign(both['masked_score'])).mean()
    mad = (both['llm_score'] - both['masked_score']).abs().mean()
    print(f'rows compared: {len(both)}')
    print(f'same sentiment label (original vs masked): {same_label:.1%}')
    print(f'same score sign: {same_sign:.1%} | mean |score difference|: {mad:.3f}')
    print('\nlabel changes (rows = original, columns = masked):')
    display(pd.crosstab(both['sentiment'], both['masked_sentiment']))

    # information: pooled Spearman of each score with the next-day abnormal return, paired day-cluster bootstrap of the difference
    def ic_pair(d):
        return (spearmanr(d['llm_score'], d['abn_ret']).correlation, spearmanr(d['masked_score'], d['abn_ret']).correlation)
    ic_o, ic_m = ic_pair(both)
    days = [g for _, g in both.groupby('entry_date')]
    rng = np.random.default_rng(SEED)
    diffs = []
    for _ in range(N_BOOT):
        d = pd.concat([days[i] for i in rng.integers(0, len(days), len(days))])
        o, m = ic_pair(d)
        diffs.append(o - m)
    lo, hi = np.nanpercentile(diffs, [2.5, 97.5])
    print(f'\npooled IC, original headline: {ic_o:+.3f} | masked, no ticker: {ic_m:+.3f}')
    print(f'difference (original - masked): {ic_o - ic_m:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]')
    print(f'smallest IC this sample size can tell from 0: about {2 / np.sqrt(len(both)):.2f}')
else:
    print('not enough masked answers to compare yet')

rows compared: 300
same sentiment label (original vs masked): 86.7%
same score sign: 88.0% | mean |score difference|: 0.128

label changes (rows = original, columns = masked):


masked_sentiment,negative,neutral,positive
sentiment,,,
negative,86,9,1
neutral,7,50,8
positive,1,14,124



pooled IC, original headline: +0.023 | masked, no ticker: +0.065
difference (original - masked): -0.042  95% CI [-0.093, +0.005]
smallest IC this sample size can tell from 0: about 0.12


## 6. The most confident wrong calls

Numbers hide what kind of mistakes the model makes. These are the rows where the LLM was most sure of the direction and the
stock went the other way, ranked by $|s|\cdot c$. We read them looking for a pattern (wrong *kind* of headline, wrong
horizon, multi-company roundups), not to fix individual rows.

In [13]:
wrong = llm[(llm['llm_score'] != 0) & (np.sign(llm['llm_score']) != np.sign(llm['abn_ret']))].copy()
wrong['strength'] = wrong['llm_score'].abs() * wrong['confidence']
cols = ['entry_date', 'ticker', 'headline', 'llm_score', 'confidence', 'event_type', 'abn_ret']
top = wrong.sort_values('strength', ascending=False).head(12)[cols].copy()
top['abn_ret'] = (top['abn_ret'] * 100).round(2).astype(str) + '%'
top['entry_date'] = top['entry_date'].dt.date
top['headline'] = top['headline'].str.slice(0, 110)
with pd.option_context('display.max_colwidth', 120):
    display(top.reset_index(drop=True))
print(f'wrong-direction rows: {len(wrong):,} of {int((llm["llm_score"] != 0).sum()):,} non-neutral rows')

,entry_date,ticker,headline,llm_score,confidence,event_type,abn_ret
0,2020-04-02,FXI,WHO's Tedros Says In The Past 5 Weeks There Has Been A 'Near Exponential Growth' In Number Of New Cases And Nu,-0.8,0.9,macro,0.11%
1,2020-02-07,CI,"Cigna Earlier Reported Q4 EPS $4.31 Beats $4.2 Estimate, Sales $36.538B Beat $35.07B Estimate; Sees FY20 EPS $",0.8,0.9,earnings,-0.08%
2,2020-01-24,JBLU,JetBlue Airways shares are trading higher after the company reported better-than-expected Q4 EPS and sales res,0.8,0.9,earnings,-1.35%
3,2020-03-23,AAL,S&P Downgrades American Air From BB- To B; Outlook Negative,-0.8,0.9,other,26.74%
4,2020-01-24,LUV,737 MAX Crisis Undercuts Southwest Airlines' Profits,-0.8,0.9,earnings,1.07%
5,2020-05-07,REGN,"Credit Suisse Maintains Outperform on Regeneron Pharmaceuticals, Raises Price Target to $605",0.8,0.9,analyst_rating,-1.02%
6,2020-02-24,DE,Deere shares are trading higher after the company reported better-than-expected Q1 EPS and sales results.,0.8,0.9,earnings,-0.0%
7,2020-01-27,TSLA,CFRA Analyst To Tesla Investors: Sell Your Stock,-0.8,0.9,analyst_rating,0.54%
8,2020-05-06,MRK,Why Shares Of Gene Therapy Company Genprex Are Surging Higher,0.8,0.9,other,-3.18%
9,2020-01-14,BIIB,"Biogen To Acquire PF-05251749, A Novel Clinical Stage Asset With Application in Alzheimer's And Parkinson's Fr",0.7,0.9,m&a,-2.19%


wrong-direction rows: 1,095 of 2,145 non-neutral rows


## 7. Robustness: headlines the source tags to a single ticker

In notebook 01 the roundup filter removed headlines that sit on 20 or more tickers, but a lot of headlines sit on 2 to 19. Some are
market roundups ("Benzinga Pro's Top 5 Stocks To Watch"). Some are about one company but tagged to related tickers (a Walmart
earnings preview shows up under other retailers). For those rows the headline may say little about the ticker whose return is the
target, which adds noise to every signal.

**The check (decided before looking at any result):** split the rows by how many tickers the source attaches to the headline,
counted over the whole raw file, and compute the same within-day IC for `n_tickers == 1` and `n_tickers >= 2`.

This is a robustness check, not a replacement for the main result. I did not pick the split after seeing the numbers, and the main
tables above keep every row. If the single-ticker rows looked clearly better it would be a lead worth testing on new data, not
a result to report as the finding, because it is one more cut of the same 3,000 rows.

In [14]:
# how many tickers does the source attach to each headline? Counted over the whole raw file, not just our 100 tickers
raw_all = pd.read_json(RAW / 'raw_analyst_ratings.jsonl', lines=True, dtype=False)[['headline', 'stock']]
n_tickers = raw_all.groupby('headline')['stock'].nunique()
del raw_all

llm['n_tickers'] = llm['headline'].map(n_tickers)
assert llm['n_tickers'].notna().all(), 'a scored headline is missing from the raw file'
llm['single_ticker'] = llm['n_tickers'] == 1
print(f'rows with a single-ticker headline: {llm["single_ticker"].sum():,} of {len(llm):,} ({llm["single_ticker"].mean():.0%})')
print(llm['n_tickers'].clip(upper=6).value_counts().sort_index().rename('rows (6 = 6 or more tickers)').to_string())

groups = [('all rows (main analysis)', pd.Series(True, index=llm.index)),
          ('single-ticker headlines', llm['single_ticker']),
          ('headlines on 2+ tickers', ~llm['single_ticker'])]
rows = []
for gname, mask in groups:
    for sname, col in [('LLM score', 'llm_score'), ('FinBERT', 'finbert')]:
        r = subset_cross_section_ic(llm, mask, col, n_boot=N_BOOT, seed=SEED)
        rows.append({'rows': gname, 'signal': sname, 'n': r['n'], 'days': r['n_days'], 'IC': r['ic'], 'CI low': r['lo'], 'CI high': r['hi']})
display(pd.DataFrame(rows).set_index(['rows', 'signal']).round(4))
print('smallest IC the single-ticker group can tell from 0: about', round(2 / np.sqrt(llm['single_ticker'].sum()), 3))

rows with a single-ticker headline: 1,638 of 3,000 (55%)
n_tickers
1    1638
2     358
3     215
4     163
5     127
6     499


n  days      IC  CI low  CI high
rows                     signal                                        
all rows (main analysis) LLM score  3000   106  0.0297 -0.0220   0.0823
                         FinBERT    3000   106  0.0298 -0.0178   0.0764
single-ticker headlines  LLM score  1638   103  0.0149 -0.0441   0.0759
                         FinBERT    1638   103  0.0056 -0.0496   0.0613
headlines on 2+ tickers  LLM score  1362   106  0.0511 -0.0196   0.1219
                         FinBERT    1362   106  0.0616  0.0008   0.1187

smallest IC the single-ticker group can tell from 0: about 0.049


## Takeaways (numbers copied from the outputs above)

Sample: 3,000 test headlines scored by gpt-oss-20b (zero-shot, batched), 106 days, 100 tickers.

1. **Overall: no detectable signal.** Within-day IC is +0.030 for the LLM [-0.022, +0.082] and +0.030 for FinBERT [-0.018, +0.076].
   The LLM is not distinguishable from zero or from FinBERT.
2. **Event type: nothing that survives the multiple-comparison caution.** Every interval contains 0. The largest point estimates
   are earnings +0.11, guidance +0.12, M&A +0.12 and product +0.09, which is the pattern one would hope for, but the guidance and M&A
   groups have only 75 and 48 rows (smallest detectable IC about 0.23-0.29) and the earnings interval is [-0.035, +0.243]. It is a
   lead to test on more data, not a finding. Analyst ratings (-0.05) and legal (-0.08) point the other way, also within noise.
3. **Ablations: the choice of score does not matter.** Plain score +0.030, sign only +0.018, score x confidence +0.029, confident rows
   only +0.031, non-neutral only +0.036. All intervals contain 0. The model's confidence does not separate better from worse calls.
4. **Sub-periods: the best period is March, for both signals.** March: LLM +0.089 [-0.012, +0.183], FinBERT +0.091 [-0.002, +0.166];
   January-February +0.008 and April-June +0.001 for the LLM. Both models look alike, so this is more likely a crash-regime effect
   (very negative headlines followed by large moves) than anything specific to the LLM. Neither interval excludes 0.
5. **Leakage check (names hidden): no evidence that the name drives the signal, but the test has little power.** Masking changed
   50% of headlines. An independent check still finds a distinctive word of the company name in 2.5% of masked headlines, mostly
   generic words like "Airlines" or "Communications" in unrelated phrases. On 300 re-scored changed headlines, 86.7% kept the same
   sentiment label and the mean absolute score change was 0.13; 38 of the 40 label changes were to or from neutral. The pooled IC
   was +0.023 with the name and +0.065 without; the difference (original minus masked) is -0.042 [-0.093, +0.005]. If anything,
   hiding the name did not hurt, the opposite of what leakage through the name would predict, but 300 rows can only detect an IC
   change of about 0.12, so I do not read anything into the sign. There is also a more basic point: leakage can only inflate a
   signal that exists. Since the LLM shows no signal, there is nothing for memory to explain.
   What this does not rule out: the model recognising the event itself (a stimulus bill, a Fed announcement) rather than the
   company. That needs news published after the model's training cutoff.
   *Revision note:* the first version of this check had alias bugs (Eli Lilly, Procter & Gamble and Wells Fargo were unmasked or only
   half masked, and ETF names produced a junk alias). After fixing them the masked sample changed; the first version gave 84.0% and
   +0.025 [-0.029, +0.084] on a different 300 rows. The conclusion did not change.
6. **Errors: roughly a coin flip.** 1,095 of the 2,145 non-neutral calls were in the wrong direction (51.0%). The most confident
   misses are partly data problems: headlines about another company tagged with the wrong ticker, and same-day news whose price
   reaction had already happened before the next-day entry.

7. **Robustness: multi-ticker headlines are not what is hiding a signal.** 55% of the rows (1,638) have a headline the source
   tags to a single ticker. On those rows the within-day IC is +0.015 for the LLM [-0.044, +0.076] and +0.006 for FinBERT, lower
   than on the other 45% (LLM +0.051 [-0.020, +0.122], FinBERT +0.062 [+0.001, +0.119]). So dropping the multi-ticker rows would not
   have helped, and I kept them in the main analysis. The FinBERT interval on multi-ticker rows only just excludes 0, and it is one
   of several cuts, so I do not read it as a finding. Headlines about several tickers may carry market or sector information
   that a single-company reading misses; I did not test that.

**Overall conclusion of the project so far.** With one model, one prompt, a single 107-day test period that contains the March 2020
crash, and a mostly large-cap universe (15 of the 100 tickers are ETFs), LLM-extracted sentiment was not distinguishable from zero and not better than a lexicon or
FinBERT, before or after costs. The result is a statement about what this sample can resolve (the LLM's gross interval is about
+-75 bps wide, so an edge of a few tens of bps per day would not be resolved), not proof that LLMs carry no signal. Strengthening it needs more test days, a model or prompt that was not tuned on
this period, news after the model's cutoff, and exact timestamps.